<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/4_3_Variant_Calling_GATK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Variant Calling with GATK and Functional Annotation

**Estimated time:** 120 minutes  **Prerequisites:** completion of Labs 1–2 (concepts only).

### Learning Objectives
- Walk through the GATK best-practices workflow from pre-processing to a filtered VCF.
- Explain the purpose of duplicate marking, HaplotypeCaller, and hard-filtering.
- Annotate called variants and interpret the added biological context.
- Prioritize candidate variants using their functional consequence.

> This notebook downloads the real GATK4 toolkit (~750&nbsp;MB) directly from its GitHub release — that first cell takes 1–3 minutes depending on Colab's connection. Everything else (reference, reads, "known" variants) is generated inside the notebook, so there is nothing to upload.

## Setup
Install Java + samtools/bwa, then download GATK4 itself (a real, current release — not a simulation of the tool).

In [1]:
%%capture
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jre-headless bwa samtools > /dev/null
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.6.0.0/gatk-4.6.0.0.zip -O gatk.zip
!unzip -o -q gatk.zip
import os, glob
gatk_dir = glob.glob("gatk-4.6.0.0")[0]
os.environ["PATH"] += ":" + os.path.abspath(gatk_dir)


In [ ]:
!gatk --version


## Background

**GATK (Genome Analysis Toolkit)** is the widely used standard for germline and somatic variant discovery. The core workflow pre-processes aligned reads by marking PCR duplicates and (on real data) recalibrating base quality scores (BQSR — skipped here, since BQSR needs a database of *known* variant sites that doesn't exist for a synthetic genome), then uses **HaplotypeCaller** to perform local *de novo* assembly of active regions and calculate genotype likelihoods, producing a **VCF** (Variant Call Format) file. **Variant Quality Score Recalibration (VQSR)**, or simpler hard-filtering, then removes likely false positives. Annotation tools such as **ANNOVAR** and **SnpEff** subsequently add biological context — coding vs. intronic vs. intergenic, predicted amino-acid consequence, and population/clinical database cross-references.

## Part A — Building a Reference Genome, a "Patient" Genome, and Reads

We build a 3,000 bp reference containing one synthetic gene (a coding region from position 500–1400), then create a "sample" genome with **four hand-designed variants** of known consequence — a missense change, a synonymous change, a nonsense (stop-gained) change, and one intergenic SNP — so you can check the pipeline's output against ground truth at every step.

In [2]:
import random
random.seed(11)

BASES = "ACGT"
GENOME_LEN = 3000
CDS_START, CDS_END = 500, 1400  # 0-based, exclusive end
GENE_NAME = "simGene1"

reference = list("".join(random.choice(BASES) for _ in range(GENOME_LEN)))

def set_codon(seq_list, cds_start, codon_index, codon):
    pos = cds_start + codon_index * 3
    for i, b in enumerate(codon):
        seq_list[pos + i] = b
    return pos

# Plant three specific codons in the CDS so their mutation consequence is known in advance
missense_codon_pos   = set_codon(reference, CDS_START, 10, "AAA")  # Lys (K)
synonymous_codon_pos = set_codon(reference, CDS_START, 40, "CCT")  # Pro (P)
nonsense_codon_pos   = set_codon(reference, CDS_START, 70, "CAA")  # Gln (Q)
intergenic_pos       = 100  # well outside the CDS

reference = "".join(reference)
with open("reference.fasta", "w") as f:
    f.write(">chrSim\n")
    for i in range(0, len(reference), 70):
        f.write(reference[i:i + 70] + "\n")

print(f"Reference: {GENOME_LEN} bp, gene {GENE_NAME} spans {CDS_START+1}-{CDS_END} (1-based)")


Reference: 3000 bp, gene simGene1 spans 501-1400 (1-based)


In [3]:
CODON_TABLE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def translate_codon(codon):
    return CODON_TABLE.get(codon.upper(), 'X')

sample = list(reference)
injected = []

def inject_codon_variant(name, codon_start_pos, mutate_offset, alt_base):
    p = codon_start_pos + mutate_offset
    sample[p] = alt_base
    codon = reference[codon_start_pos:codon_start_pos + 3]
    new_codon = list(codon)
    new_codon[mutate_offset] = alt_base
    new_codon = "".join(new_codon)
    injected.append(dict(name=name, pos=p + 1, ref=reference[p], alt=alt_base,
                          orig_codon=codon, new_codon=new_codon,
                          orig_aa=translate_codon(codon), new_aa=translate_codon(new_codon)))

inject_codon_variant("missense (planted)",   missense_codon_pos,   1, "G")  # AAA(K) -> AGA(R)
inject_codon_variant("synonymous (planted)", synonymous_codon_pos, 2, "C")  # CCT(P) -> CCC(P)
inject_codon_variant("nonsense (planted)",   nonsense_codon_pos,   0, "T")  # CAA(Q) -> TAA(*)

ref_base = reference[intergenic_pos]
alt_base = random.choice([b for b in BASES if b != ref_base])
sample[intergenic_pos] = alt_base
injected.append(dict(name="intergenic (planted)", pos=intergenic_pos + 1, ref=ref_base, alt=alt_base,
                      orig_codon=None, new_codon=None, orig_aa=None, new_aa=None))

sample = "".join(sample)
with open("sample_genome.fasta", "w") as f:
    f.write(">chrSim_sample\n")
    for i in range(0, len(sample), 70):
        f.write(sample[i:i + 70] + "\n")

import pandas as pd
pd.DataFrame(injected)


,name,pos,ref,alt,orig_codon,new_codon,orig_aa,new_aa
0,missense (planted),532,A,G,AAA,AGA,K,R
1,synonymous (planted),623,T,C,CCT,CCC,P,P
2,nonsense (planted),711,C,T,CAA,TAA,Q,*
3,intergenic (planted),101,G,C,None,None,None,None


In [4]:
def mutate(seq, error_rate):
    bases = list(seq)
    for i in range(len(bases)):
        if random.random() < error_rate:
            bases[i] = random.choice([b for b in BASES if b != bases[i]])
    return "".join(bases)

def phred_string(length, mean_q=35):
    return "".join(chr(min(max(mean_q + random.randint(-3, 3), 2), 40) + 33) for _ in range(length))

READ_LEN = 100
DEPTH = 30
N_READS = int(GENOME_LEN * DEPTH / READ_LEN)

reads = []
for i in range(N_READS):
    start = random.randint(0, GENOME_LEN - READ_LEN)
    read_seq = mutate(sample[start:start + READ_LEN], 0.005)  # 0.5% sequencing error
    reads.append((f"read_{i}", read_seq, phred_string(READ_LEN)))

with open("sample_reads.fastq", "w") as f:
    for name, seq, qual in reads:
        f.write(f"@{name}\n{seq}\n+\n{qual}\n")

print(f"Simulated {len(reads)} reads from the sample genome (~{DEPTH}x coverage)")


Simulated 900 reads from the sample genome (~30x coverage)


## Part B — Pre-processing and Variant Calling

In [5]:
!bwa index reference.fasta
!bwa mem -R "@RG\tID:sample1\tSM:sample1\tPL:ILLUMINA" reference.fasta sample_reads.fastq > aln.sam 2> bwa.log
!samtools sort aln.sam -o aln.sorted.bam
!samtools index aln.sorted.bam
!samtools faidx reference.fasta
!gatk CreateSequenceDictionary -R reference.fasta -QUIET true
!samtools flagstat aln.sorted.bam


[bwa_index] Pack FASTA... 0.00 sec
[bwa_index] Construct BWT for the packed sequence...
[bwa_index] 0.00 seconds elapse.
[bwa_index] Update BWT... 0.00 sec
[bwa_index] Pack forward-only FASTA... 0.00 sec
[bwa_index] Construct SA from BWT and Occ... 0.00 sec
[main] Version: 0.7.17-r1188
[main] CMD: bwa index reference.fasta
[main] Real time: 0.057 sec; CPU: 0.007 sec
Using GATK jar /content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar CreateSequenceDictionary -R reference.fasta -QUIET true
[0.029s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.029s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../

**Q1.** What fraction of reads aligned? Given the reference is small and the reads are simulated from it directly, would you expect anything less than ~100%?

_Your answer:_



In [6]:
!gatk MarkDuplicates -I aln.sorted.bam -O aln.dedup.bam -M dup_metrics.txt --QUIET true 2>&1 | tail -5
!samtools index aln.dedup.bam
!grep -A2 "LIBRARY" dup_metrics.txt


INFO	2026-09-05 11:05:30	MarkDuplicates	Before output close freeMemory: 74515120; totalMemory: 104857600; maxMemory: 3401580544
INFO	2026-09-05 11:05:30	MarkDuplicates	Closed outputs. Getting more Memory Stats.
INFO	2026-09-05 11:05:30	MarkDuplicates	After output close freeMemory: 74989848; totalMemory: 104857600; maxMemory: 3401580544
Tool returned:
0
LIBRARY	UNPAIRED_READS_EXAMINED	READ_PAIRS_EXAMINED	SECONDARY_OR_SUPPLEMENTARY_RDS	UNMAPPED_READS	UNPAIRED_READ_DUPLICATES	READ_PAIR_DUPLICATES	READ_PAIR_OPTICAL_DUPLICATES	PERCENT_DUPLICATION	ESTIMATED_LIBRARY_SIZE
Unknown Library	900	0	0	0	126	0	0	0.14	



**Q2.** Look at the `PERCENT_DUPLICATION` column in the metrics above. Why is duplicate marking important before variant calling, and what artifact would appear in a real dataset if it were skipped?

_Your answer:_



**Note on BQSR:** on real data, the next best-practices step is Base Quality Score Recalibration (`BaseRecalibrator` + `ApplyBQSR`), which corrects systematic errors in the sequencer's own quality estimates using a database of known variant sites (e.g. dbSNP). Our synthetic genome has no such database, so we skip it here — but you should run it on any real dataset.

In [7]:
!gatk HaplotypeCaller -R reference.fasta -I aln.dedup.bam -O raw.vcf.gz --QUIET true


Using GATK jar /content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar HaplotypeCaller -R reference.fasta -I aln.dedup.bam -O raw.vcf.gz --QUIET true
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
11:05:43.919 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
11:05:44.283 INFO  HaplotypeCaller - Initializing engine
11:05:44.574 INFO  HaplotypeCaller - Done initializing

In [9]:
!pip install pysam
import pysam

vcf = pysam.VariantFile("raw.vcf.gz")
rows = []
for rec in vcf:
    rows.append(dict(CHROM=rec.chrom, POS=rec.pos, REF=rec.ref, ALT=rec.alts[0],
                      QUAL=round(rec.qual, 1), DP=rec.info.get("DP"),
                      GT=rec.samples[0]["GT"]))
import pandas as pd
raw_df = pd.DataFrame(rows)
raw_df

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 22.9 MB/s eta 0:00:00


,CHROM,POS,REF,ALT,QUAL,DP,GT
0,chrSim,101,G,C,976.1,27,"(1, 1)"
1,chrSim,532,A,G,1052.1,27,"(1, 1)"
2,chrSim,623,T,C,735.1,19,"(1, 1)"
3,chrSim,711,C,T,930.1,26,"(1, 1)"


**Q3.** Compare the positions in `raw_df` to the `pos` column of the injected-variants table from Part A. Did HaplotypeCaller recover all four planted variants?

_Your answer:_



## Part C — Filtering
With only four true variants and no sequencing artifacts deliberately introduced, everything here should pass — but the thresholds below are the standard GATK hard-filter starting point, and you should see how they behave.

In [10]:
!gatk VariantFiltration -R reference.fasta -V raw.vcf.gz \
    --filter-expression "QD < 2.0"  --filter-name "lowQD" \
    --filter-expression "MQ < 40.0" --filter-name "lowMQ" \
    --filter-expression "FS > 60.0" --filter-name "highFS" \
    -O filtered.vcf.gz --QUIET true


Using GATK jar /content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar VariantFiltration -R reference.fasta -V raw.vcf.gz --filter-expression QD < 2.0 --filter-name lowQD --filter-expression MQ < 40.0 --filter-name lowMQ --filter-expression FS > 60.0 --filter-name highFS -O filtered.vcf.gz --QUIET true
[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
11:07:04.541 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.0.0/gatk-package-4.6.0.0-local.jar!/

In [11]:
vcf = pysam.VariantFile("filtered.vcf.gz")
rows = []
for rec in vcf:
    rows.append(dict(CHROM=rec.chrom, POS=rec.pos, REF=rec.ref, ALT=rec.alts[0],
                      QUAL=round(rec.qual, 1), DP=rec.info.get("DP"),
                      FILTER=";".join(rec.filter.keys())))
filtered_df = pd.DataFrame(rows)
filtered_df


,CHROM,POS,REF,ALT,QUAL,DP,FILTER
0,chrSim,101,G,C,976.1,27,PASS
1,chrSim,532,A,G,1052.1,27,PASS
2,chrSim,623,T,C,735.1,19,PASS
3,chrSim,711,C,T,930.1,26,PASS


**Q4.** Did any variant fail a filter? If you lower `--filter-expression "QD < 2.0"` to a stricter `"QD < 30.0"`, what happens, and why does that illustrate the trade-off between sensitivity and specificity in variant filtering?

_Your answer:_



## Part D — Functional Annotation

Real pipelines would hand the filtered VCF to **ANNOVAR** or **SnpEff** together with a genome annotation database. Since our reference is a synthetic, single-gene toy genome with no public annotation database behind it, we annotate it ourselves with a small Python function that captures exactly what those tools do conceptually: decide whether a variant falls in a coding region, translate the affected codon, and classify the consequence.

In [12]:
def annotate_variant(pos1, ref_base, alt_base, cds_start=CDS_START, cds_end=CDS_END, gene=GENE_NAME):
    pos0 = pos1 - 1
    if not (cds_start <= pos0 < cds_end):
        return dict(gene="-", region="intergenic", consequence="intergenic", aa_change="-")
    offset = (pos0 - cds_start) % 3
    codon_start = pos0 - offset
    codon = reference[codon_start:codon_start + 3]
    new_codon = list(codon)
    new_codon[offset] = alt_base
    new_codon = "".join(new_codon)
    aa_ref, aa_alt = translate_codon(codon), translate_codon(new_codon)
    codon_number = (codon_start - cds_start) // 3 + 1
    if aa_ref == aa_alt:
        consequence = "synonymous"
    elif aa_alt == "*":
        consequence = "nonsense"
    else:
        consequence = "missense"
    return dict(gene=gene, region="CDS", consequence=consequence,
                aa_change=f"{aa_ref}{codon_number}{aa_alt}")

annotations = []
for _, row in filtered_df.iterrows():
    ann = annotate_variant(row["POS"], row["REF"], row["ALT"])
    annotations.append({**row.to_dict(), **ann})

annotation_table = pd.DataFrame(annotations)
annotation_table


,CHROM,POS,REF,ALT,QUAL,DP,FILTER,gene,region,consequence,aa_change
0,chrSim,101,G,C,976.1,27,PASS,-,intergenic,intergenic,-
1,chrSim,532,A,G,1052.1,27,PASS,simGene1,CDS,missense,K11R
2,chrSim,623,T,C,735.1,19,PASS,simGene1,CDS,synonymous,P41P
3,chrSim,711,C,T,930.1,26,PASS,simGene1,CDS,nonsense,Q71*


**Q5.** For each variant, record: gene symbol, functional consequence, and amino-acid change (or '-' for intergenic).

_Your answer:_



In [13]:
rank_order = {"nonsense": 0, "missense": 1, "synonymous": 2, "intergenic": 3}
annotation_table["priority_rank"] = annotation_table["consequence"].map(rank_order)
annotation_table.sort_values("priority_rank").drop(columns="priority_rank")


,CHROM,POS,REF,ALT,QUAL,DP,FILTER,gene,region,consequence,aa_change
3,chrSim,711,C,T,930.1,26,PASS,simGene1,CDS,nonsense,Q71*
1,chrSim,532,A,G,1052.1,27,PASS,simGene1,CDS,missense,K11R
2,chrSim,623,T,C,735.1,19,PASS,simGene1,CDS,synonymous,P41P
0,chrSim,101,G,C,976.1,27,PASS,-,intergenic,intergenic,-


**Q6.** Does the ranking above match your own judgment of biological importance? Would you rank the synonymous variant above or below the intergenic one, and why?

_Your answer:_



## Discussion Questions

**Q7.** Why does HaplotypeCaller perform local re-assembly rather than simply reading off the pileup at each position?

_Your answer:_



**Q8.** In a cancer genomics context, why might a variant with a high population frequency in a database like gnomAD be deprioritized as a candidate driver mutation, even if our simple ranking above doesn't account for that?

_Your answer:_



**Q9.** Our custom `annotate_variant` function is a simplified stand-in for ANNOVAR/SnpEff. Name two things a real annotation tool does that ours does not.

_Your answer:_



---
### Deliverable
The filtered VCF table, the final annotation table with your priority ranking, and written answers to Parts A–D and the discussion questions.